In [22]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.utils import shuffle
from scipy.signal import resample
import matplotlib.pyplot as plt
import json
import warnings
import mne
from mne.preprocessing import ICA
warnings.filterwarnings("ignore")
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [23]:
SAMPLE_RATE = 200  # fs
# baseline correction according to paper, tmin and tmax was observed from the data
TASK_ID_1 = 4   # MSIT
TMIN_1, TMAX_1 = -0.5, 1.0
BASELINE_1 = (-0.5, 0)   

TASK_ID_2 = 1   # Visual Oddball
TMIN_2, TMAX_2 = -0.5, 1.0
BASELINE_2 = (-0.5, 0)

TASK_ID_3 = 9   # SRT
TMIN_3, TMAX_3 = -0.2, 0.8
BASELINE_3 = (-0.2, 0)

sub_folder_path = str(SAMPLE_RATE) + 'Hz'
sub_folder_path

'200Hz'

In [24]:
## Load participants.tsv file
# root dirNencki-Symfonia EEG
root = "NSERP/"
participants_path = os.path.join(root, "participants.tsv")
participants = pd.read_csv(participants_path, sep="\t")
participants

,participant_id,sex,age,medication use,medication type,caffeine uptake,stress,rest,menstrual cycle,UMACL/UWIST Niezadowolony Dissatisfied,...,ARSQ_41,ARSQ_42,ARSQ_43,ARSQ_44,ARSQ_45,ARSQ_46,ARSQ_47,ARSQ_48,ARSQ_49,ARSQ_50
0,sub-01,F,22,1,ethinylestradiolum/ drospirenonum,0,2,4,10.0,4,...,1,2,4,1,2,3,4,4,2,5
1,sub-02,F,21,0,NaN,1,5,3,21.0,3,...,1,1,1,1,1,4,1,3,1,5
2,sub-03,M,27,0,NaN,0,3,5,NaN,3,...,1,1,5,1,1,3,2,2,2,5
3,sub-04,M,26,0,NaN,0,3,3,NaN,3,...,1,1,1,1,1,3,3,3,3,5
4,sub-05,M,25,0,NaN,0,4,1,NaN,3,...,1,1,1,1,4,4,2,2,1,5
5,sub-06,M,26,0,NaN,0,4,3,NaN,3,...,1,1,1,3,2,1,3,1,1,4
6,sub-07,M,25,0,NaN,0,1,3,NaN,2,...,1,2,1,1,1,4,5,5,2,5
7,sub-08,M,33,1,levothyroxinum natricum,1,2,4,NaN,4,...,1,1,4,1,1,2,3,2,2,5
8,sub-09,F,20,0,NaN,0,4,2,20.0,3,...,2,1,1,1,1,3,1,1,1,5
9,sub-10,F,20,0,NaN,0,4,3,16.0,4,...,1,1,1,1,1,4,3,1,2,5


In [25]:
# Test for bad channels, sampling freq and shape
# The data is BrainVision format (Brain Products) not EEGLAB .set/.fdt format

"""bad_channel_list, sampling_freq_list, data_shape_list = [], [], []

# Loop through each subject folder
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg')
        if not os.path.exists(sub_path):
            print(f"❌ Missing folder: {sub_path}")
            continue
        files = os.listdir(sub_path)
        # Look for the .vhdr file (BrainVision header file)
        for file in files:
            if file.endswith('.vhdr'):
                file_path = os.path.join(sub_path, file)
                print(f"✅ Reading: {file_path}")
                raw = mne.io.read_raw_brainvision(file_path, preload=True)

                # Collect bad channels
                bad_channel_list.append(raw.info['bads'])
                # Collect sampling frequency
                sampling_freq_list.append(raw.info['sfreq'])
                # Collect EEG data shape
                data_shape_list.append(raw.get_data().shape)"""


'bad_channel_list, sampling_freq_list, data_shape_list = [], [], []\n\n# Loop through each subject folder\nfor sub in os.listdir(root):\n    if \'sub-\' in sub:\n        sub_path = os.path.join(root, sub, \'eeg\')\n        if not os.path.exists(sub_path):\n            print(f"❌ Missing folder: {sub_path}")\n            continue\n        files = os.listdir(sub_path)\n        # Look for the .vhdr file (BrainVision header file)\n        for file in files:\n            if file.endswith(\'.vhdr\'):\n                file_path = os.path.join(sub_path, file)\n                print(f"✅ Reading: {file_path}")\n                raw = mne.io.read_raw_brainvision(file_path, preload=True)\n\n                # Collect bad channels\n                bad_channel_list.append(raw.info[\'bads\'])\n                # Collect sampling frequency\n                sampling_freq_list.append(raw.info[\'sfreq\'])\n                # Collect EEG data shape\n                data_shape_list.append(raw.get_data().shape)'

In [5]:
"""from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
(127, 1437860)
Channel number counter: Counter({127: 167})
Sampling rate counter: Counter({1000.0: 167})


In [6]:
"""# Initialize common channel list
common_channels = []

# Loop through subject folders
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        for file in os.listdir(sub_path):
            # Change: look for BrainVision .vhdr files
            if file.endswith('.vhdr'):
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_brainvision(file_path, preload=True)
                current_channels = set(raw.info['ch_names'])

                if not common_channels:
                    common_channels = current_channels
                else:
                    common_channels &= current_channels  # Intersection

# Convert set to list
common_channels = list(common_channels)
# Print result
print(common_channels)
print("Common channels number: ", len(common_channels))"""

Extracting parameters from NSERP/sub-01\eeg/sub-01_task-msit_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 1437859  =      0.000 ...  1437.859 secs...
Extracting parameters from NSERP/sub-01\eeg/sub-01_task-oddball_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 1371319  =      0.000 ...  1371.319 secs...
Extracting parameters from NSERP/sub-01\eeg/sub-01_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 732699  =      0.000 ...   732.699 secs...
Extracting parameters from NSERP/sub-01\eeg/sub-01_task-srt_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 428819  =      0.000 ...   428.819 secs...
Extracting parameters from NSERP/sub-02\eeg/sub-02_task-msit_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 1308539  =      0.000 ...  1308.539 secs...
Extracting parameters from NSERP/sub-02\eeg/sub-02_task-oddball_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 1345819  =      0.000 ...  1345.819 secs...
Extract

In [26]:
common_channels = ['FC6', 'PO10', 'TP10 RIGHT EAR', 'F2', 'AFF2h', 'F6', 'FCC5h', 'FC2', 'TP8', 'FCC2h', 'POO2', 'F9', 'CP4', 'P6', 'CP5', 'C6', 'F10', 'POO10h', 'Fz', 'POO1', 'POz', 'AF7', 'OI1h', 'AFp1', 'FFC5h', 'AFz', 'CP2', 'AF3', 'CCP2h', 'PPO9h', 'O1', 'POO9h', 'PO8', 'Fp1', 'P2', 'FTT10h', 'Oz', 'P8', 'CP3', 'FCC1h', 'TTP8h', 'Fp2', 'Cz', 'CP6', 'FT7', 'AFF5h', 'FTT8h', 'CP1', 'PO3', 'CPP6h', 'FTT7h', 'PPO1h', 'TP9 LEFT EAR', 'P5', 'FC4', 'PO4', 'FC3', 'CPP4h', 'CPP5h', 'PPO10h', 'F1', 'CCP5h', 'T7', 'C1', 'AF4', 'FFT7h', 'Pz', 'TPP7h', 'FFT9h', 'CPP1h', 'AF8', 'FCC6h', 'T8', 'O10', 'AFp2', 'PPO5h', 'PO9', 'FFC6h', 'F3', 'C4', 'O9', 'P1', 'F7', 'TP7', 'FFC4h', 'C5', 'FFC2h', 'P9', 'FT10', 'FCC4h', 'PPO2h', 'FC5', 'TPP8h', 'P7', 'FT9', 'TPP10h', 'CCP4h', 'FFT10h', 'CPz', 'F5', 'FFC1h', 'FC1', 'CPP2h', 'P10', 'CPP3h', 'FFC3h', 'PO7', 'AFF1h', 'P4', 'C2', 'OI2h', 'CCP3h', 'PPO6h', 'P3', 'TTP7h', 'CCP1h', 'FFT8h', 'CCP6h', 'FT8', 'O2', 'F8', 'AFF6h', 'F4', 'FCC3h', 'FTT9h', 'TPP9h', 'C3']

In [27]:
# remove unreliable electrodes 
common_channels = [ch for ch in common_channels if ch not in ['TP9 LEFT EAR', 'TP10 RIGHT EAR', 'O9', 'O10']]
print(common_channels)
print("Common channels number: ", len(common_channels))

['FC6', 'PO10', 'F2', 'AFF2h', 'F6', 'FCC5h', 'FC2', 'TP8', 'FCC2h', 'POO2', 'F9', 'CP4', 'P6', 'CP5', 'C6', 'F10', 'POO10h', 'Fz', 'POO1', 'POz', 'AF7', 'OI1h', 'AFp1', 'FFC5h', 'AFz', 'CP2', 'AF3', 'CCP2h', 'PPO9h', 'O1', 'POO9h', 'PO8', 'Fp1', 'P2', 'FTT10h', 'Oz', 'P8', 'CP3', 'FCC1h', 'TTP8h', 'Fp2', 'Cz', 'CP6', 'FT7', 'AFF5h', 'FTT8h', 'CP1', 'PO3', 'CPP6h', 'FTT7h', 'PPO1h', 'P5', 'FC4', 'PO4', 'FC3', 'CPP4h', 'CPP5h', 'PPO10h', 'F1', 'CCP5h', 'T7', 'C1', 'AF4', 'FFT7h', 'Pz', 'TPP7h', 'FFT9h', 'CPP1h', 'AF8', 'FCC6h', 'T8', 'AFp2', 'PPO5h', 'PO9', 'FFC6h', 'F3', 'C4', 'P1', 'F7', 'TP7', 'FFC4h', 'C5', 'FFC2h', 'P9', 'FT10', 'FCC4h', 'PPO2h', 'FC5', 'TPP8h', 'P7', 'FT9', 'TPP10h', 'CCP4h', 'FFT10h', 'CPz', 'F5', 'FFC1h', 'FC1', 'CPP2h', 'P10', 'CPP3h', 'FFC3h', 'PO7', 'AFF1h', 'P4', 'C2', 'OI2h', 'CCP3h', 'PPO6h', 'P3', 'TTP7h', 'CCP1h', 'FFT8h', 'CCP6h', 'FT8', 'O2', 'F8', 'AFF6h', 'F4', 'FCC3h', 'FTT9h', 'TPP9h', 'C3']
Common channels number:  123


In [28]:
def data_preprocessing(
    raw: mne.io.Raw,
    common_channels: list,
    sample_rate: int = 250,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      1) choose common channels and reorder
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（on 1 Hz high-pass filtered copy）
      8) downsample to 250 Hz
    """

    # Remove 'boundary' or 'segment' events before anything else
    # Events like "boundary" or "new segment" inserted during file conversion are not real trials and must be excluded from annotations before epoching.
    raw.annotations.delete([
        i for i, desc in enumerate(raw.annotations.description)
        if 'boundary' in desc.lower() or 'segment' in desc.lower()
    ])
    if verbose:
        print("✔ Cleaned: Removed 'boundary' or 'segment' annotations")

    # 1. select common channels and reorder to given order
    keep = [ch for ch in common_channels if ch in raw.ch_names]
    raw.pick_channels(keep)
    raw.reorder_channels(keep)
    if verbose:
        print(f"✔ Step 1: Picked common channels ({len(keep)}): {keep}")

    # Set standard 10-05 montage 
    raw.set_montage('standard_1005')
    if verbose:
        print("✔ Step 2: Set standard 10-05 montage")
        
    # 3. Notch
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")
            
    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")

    print()
    
    # 7) ICA (fit ICLabel on 1 Hz high-pass filtered copy, then apply to original)
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, picks="eeg", verbose=False)
    ica = ICA(n_components=None, method="fastica", random_state=97, max_iter="auto")
    ica.fit(raw_for_ica)

    excluded = []
    if 'label_components' in globals() and label_components is not None:
        try:
            ic_labels = label_components(raw_for_ica, ica, method="iclabel")
            labels = ic_labels["labels"]
            probs = ic_labels["y_pred_proba"]
            thresholds = {
                "eye blink": 0.7,
                "muscle artifact": 0.6,
                "heart beat": 0.5,
                "line noise": 0.8,
                "channel noise": 0.9,
            }
            for i, lab in enumerate(labels):
                if lab in thresholds:
                    p = probs[i].max() if probs is not None else 1.0
                    if p >= thresholds[lab]:
                        excluded.append(i)
        except Exception as e:
            if verbose:
                print(f"⚠ ICLabel failed ({e}). Skipping auto exclusion.")
    else:
        if verbose:
            print("ℹ ICLabel not available; fitted ICA but no auto component exclusion.")

    if excluded:
        ica.exclude = sorted(set(excluded))
        raw = ica.apply(raw.copy())
        if verbose:
            print(f"✔ Step 7: ICA applied. Excluded comps: {ica.exclude}")
    else:
        if verbose:
            print("ℹ Step 7: No ICA components excluded.")

    # 8) downsample to 250 Hz
    if raw.info["sfreq"] != sample_rate:
        raw.resample(sample_rate, npad="auto", verbose=False)
    if verbose:
        print(f"✔ Step 8: Resampled to {sample_rate} Hz")
        
    return raw


In [29]:
mapping_dict = {
    # SRT: only 5; 
    # Oddball: 5,6,7 for standard,target,deviant; 
    # MSIT: 5,6,7,8 for F0, FS, 00, S0 according to paper
    5:0, 6:1, 7:2, 8:3
}


def epoch_and_make_xy(
    raw: mne.io.Raw,
    events_tsv_path: str,
    tmin: float = -0.5,
    tmax: float = 1.5,
    baseline=(-0.5, 0),
    task_id: int = 1,
    subject_id: int = 1,
):

    # Read and parse events.tsv
    ev = pd.read_csv(events_tsv_path, sep="\t")
    ev = ev[ev["event_type"].astype(str).str.match(r"^S\s*\d+$")].copy()
    ev["code"] = ev["event_type"].str.extract(r"(\d+)").astype(int)

    # Select stimulus events based on task type
    stim = ev[(ev["code"].isin([5, 6, 7, 8])) & (ev["trial_type"] == "stimulus")].reset_index(drop=True)
    labels = stim["code"].map(mapping_dict).astype(int)
    labels = labels.values

    sfreq = raw.info["sfreq"]
    print(f"Current sampling frequency: {sfreq} Hz")

    # Convert stimulus events to sample indices
    onset_samples = np.round(stim["onset"].values * sfreq).astype(int)

    # Check epoch time boundaries to avoid exceeding raw data limits
    min_sample = int(-tmin * sfreq)
    max_sample = int(len(raw) - tmax * sfreq)
    valid_mask = (onset_samples >= min_sample) & (onset_samples <= max_sample)
    onset_samples = onset_samples[valid_mask]
    labels = labels[valid_mask]
    events = np.c_[onset_samples, np.zeros_like(onset_samples), np.ones_like(onset_samples).astype(int)]

    # Extract epochs
    picks = mne.pick_types(raw.info, eeg=True, eog=False, exclude="bads")
    epochs = mne.Epochs(
        raw, events, event_id=dict(stim=1),
        tmin=tmin, tmax=tmax,
        baseline=baseline, picks=picks,
        proj=False, preload=True, reject=None, verbose=False
    )

    # Get indices of valid trials
    good_idx = epochs.selection  # Map indices back to positions in stim/labels
    labels = labels[good_idx]    # Trim labels to match EEG epochs

    # Get EEG data (N, C, T)
    target_len = int((tmax - tmin) * sfreq)
    data = epochs.get_data()
    if data.shape[-1] > target_len:
        data = data[..., :target_len]
    elif data.shape[-1] < target_len:
        pad = target_len - data.shape[-1]
        data = np.pad(data, ((0, 0), (0, 0), (0, pad)), mode="edge")

    # Transpose to (N, T, C)
    X = np.transpose(data, (0, 2, 1))
    
    # Create label array y with structure: [disease_id, stimulus_type, subject_id, task_id]
    y = np.column_stack([
        np.full_like(labels, -1),  # disease_id is not provided, set to -1
        labels,
        np.full_like(labels, subject_id),
        np.full_like(labels, task_id),
    ])
    

    return X, y

In [30]:
import re
import os

def get_subject_id(path: str) -> int:
    # match string like "sub-xxx" 
    match = re.search(r"sub-(\d+)", path)
    if match:
        return int(match.group(1).lstrip("0"))  
    else:
        raise ValueError(f"No subject_id found in path: {path}")

## PASS 1: Find total number of trials across all subjects

In [25]:
N_total_oddball, N_total_msit, N_total_srt = 0, 0, 0
task_id, tmin, tmax, baseline = None, None, None, None
    
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        subject_id = get_subject_id(sub_path)
        
        for file in os.listdir(sub_path):
            if 'msit_eeg.vhdr' in file or 'oddball_eeg.vhdr' in file or "srt_eeg.vhdr" in file: 
                # check task
                # only use visual task here
                if "msit_eeg.vhdr" in file:
                    task_id = TASK_ID_1
                    tmin = TMIN_1
                    tmax = TMAX_1
                    baseline = BASELINE_1
                if "oddball_eeg.vhdr" in file: 
                    task_id = TASK_ID_2
                    tmin = TMIN_2
                    tmax = TMAX_2
                    baseline = BASELINE_2
                if "srt_eeg.vhdr" in file: 
                    task_id = TASK_ID_3
                    tmin = TMIN_3
                    tmax = TMAX_3
                    baseline = BASELINE_3
                # load eeg data and preprocess
                print(sub_path)
                
                vhdr_path = os.path.join(sub_path, file)
                events_file = file.split(".")[0].split("_")[0] + "_" + file.split(".")[0].split("_")[1] + "_events.tsv"
                events_file_path = os.path.join(sub_path, events_file)
                
                print(vhdr_path)
                raw = mne.io.read_raw_brainvision(vhdr_path, preload=True)                
                raw.pick_types(eeg=True)   # only keep EEG channels
                # raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=50, l_freq=0.5, h_freq=40, verbose=True)
                print()
                
                # segment and make X, y
                
                print(f"Start epoching file {file} making X, y...")
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=tmin, tmax=tmax, baseline=baseline,  #  -200ms - 800ms
                    task_id=task_id, subject_id=subject_id, 
                )
                print(f"File {file} epoch into trial X shape: {X.shape}, y shape: {y.shape}")
                
                if task_id == TASK_ID_1:
                    N_total_msit += X.shape[0]
                elif task_id == TASK_ID_2:
                    N_total_oddball += X.shape[0]
                elif task_id == TASK_ID_3:
                    N_total_srt += X.shape[0]
                
                print("-----------------------\n")

            print("------------------------------------------------\n")

print("Total trials across all subjects in MSIT:", N_total_msit)
print("Total trials across all subjects in Oddball:", N_total_oddball)
print("Total trials across all subjects in SRT:", N_total_srt)

------------------------------------------------

------------------------------------------------

NSERP/sub-01\eeg/
NSERP/sub-01\eeg/sub-01_task-msit_eeg.vhdr
Extracting parameters from NSERP/sub-01\eeg/sub-01_task-msit_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 1437859  =      0.000 ...  1437.859 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).

Start epoching file sub-01_task-msit_eeg.vhdr making X, y...
Current sampling frequency: 1000.0 Hz
File sub-01_task-msit_eeg.vhdr epoch into trial X shape: (406, 1500, 127), y shape: (406, 4)
-----------------------

------------------------------------------------

------------------------------------------------

------------------------------------------------

------------------------------------------------

------------------------------------------------

NSERP/sub-01\eeg/
NSERP/sub-01\eeg/sub-01_task-oddball_eeg.vhdr
Extracting parameters from NSERP/sub-01\eeg/sub-01_task-oddball_e

In [10]:
"""N_total_msit, N_total_oddball, N_total_srt = 16729, 27865, 5288
print("Total trials across all subjects in MSIT:", N_total_msit)
print("Total trials across all subjects in Oddball:", N_total_oddball)
print("Total trials across all subjects in SRT:", N_total_srt)"""

Total trials across all subjects in MSIT: 16729
Total trials across all subjects in Oddball: 27865
Total trials across all subjects in SRT: 5288


## Create memmap storage for each task

In [31]:
output_root_msit = os.path.join('Processed', sub_folder_path, 'NSERP-MSIT')
os.makedirs(output_root_msit, exist_ok=True)

X_path_msit = os.path.join(output_root_msit, 'X.dat')
y_path_msit = os.path.join(output_root_msit, 'y.dat')
meta_path_msit = os.path.join(output_root_msit, 'meta.json')

print("X path of msit task:", X_path_msit)
print("y path of msit task:", y_path_msit)

# create memmap storage
SEG_LEN_MSIT = int((TMAX_1 - TMIN_1) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm_msit = np.memmap(X_path_msit, dtype='float32', mode='w+', shape=(N_total_msit, SEG_LEN_MSIT, n_channels))
y_mm_msit = np.memmap(y_path_msit, dtype='float32', mode='w+', shape=(N_total_msit, 4))
print("Created memmap for MSIT: X shape =", X_mm_msit.shape, ", y shape =", y_mm_msit.shape)

X path of msit task: Processed\200Hz\NSERP-MSIT\X.dat
y path of msit task: Processed\200Hz\NSERP-MSIT\y.dat
Created memmap for MSIT: X shape = (16729, 300, 123) , y shape = (16729, 4)


In [32]:
output_root_odd = os.path.join('Processed', sub_folder_path, 'NSERP-ODD')
os.makedirs(output_root_odd, exist_ok=True)

X_path_odd = os.path.join(output_root_odd, 'X.dat')
y_path_odd = os.path.join(output_root_odd, 'y.dat')
meta_path_odd = os.path.join(output_root_odd, 'meta.json')

print("X path of oddball task:", X_path_odd)
print("y path of oddball task:", y_path_odd)

# create memmap storage
SEG_LEN_ODD = int((TMAX_2 - TMIN_2) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm_odd = np.memmap(X_path_odd, dtype='float32', mode='w+', shape=(N_total_oddball, SEG_LEN_ODD, n_channels))
y_mm_odd = np.memmap(y_path_odd, dtype='float32', mode='w+', shape=(N_total_oddball, 4))
print("Created memmap for Oddball: X shape =", X_mm_odd.shape, ", y shape =", y_mm_odd.shape)

X path of oddball task: Processed\200Hz\NSERP-ODD\X.dat
y path of oddball task: Processed\200Hz\NSERP-ODD\y.dat
Created memmap for Oddball: X shape = (27865, 300, 123) , y shape = (27865, 4)


In [33]:
output_root_srt = os.path.join('Processed', sub_folder_path, 'NSERP-SRT')
os.makedirs(output_root_srt, exist_ok=True)

X_path_srt = os.path.join(output_root_srt, 'X.dat')
y_path_srt = os.path.join(output_root_srt, 'y.dat')
meta_path_srt = os.path.join(output_root_srt, 'meta.json')

print("X path of srt task:", X_path_srt)
print("y path of srt task:", y_path_srt)

# create memmap storage
SEG_LEN_SRT = int((TMAX_3 - TMIN_3) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm_srt = np.memmap(X_path_srt, dtype='float32', mode='w+', shape=(N_total_srt, SEG_LEN_SRT, n_channels))
y_mm_srt = np.memmap(y_path_srt, dtype='float32', mode='w+', shape=(N_total_srt, 4))
print("Created memmap for SRT: X shape =", X_mm_srt.shape, ", y shape =", y_mm_srt.shape)

X path of srt task: Processed\200Hz\NSERP-SRT\X.dat
y path of srt task: Processed\200Hz\NSERP-SRT\y.dat
Created memmap for SRT: X shape = (5288, 200, 123) , y shape = (5288, 4)


## PASS 2: Process and store trials into memmap

In [19]:
cur_msit_idx, cur_odd_idx, cur_srt_idx = 0, 0, 0
    
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        subject_id = get_subject_id(sub_path)
        
        for file in os.listdir(sub_path):
            if 'msit_eeg.vhdr' in file or 'oddball_eeg.vhdr' in file or "srt_eeg.vhdr" in file: 
                # check task
                # only use visual task here
                if "msit_eeg.vhdr" in file:
                    task_id = TASK_ID_1
                    tmin = TMIN_1
                    tmax = TMAX_1
                    baseline = BASELINE_1
                if "oddball_eeg.vhdr" in file: 
                    task_id = TASK_ID_2
                    tmin = TMIN_2
                    tmax = TMAX_2
                    baseline = BASELINE_2
                if "srt_eeg.vhdr" in file: 
                    task_id = TASK_ID_3
                    tmin = TMIN_3
                    tmax = TMAX_3
                    baseline = BASELINE_3
                # load eeg data and preprocess
                print(sub_path)
                
                vhdr_path = os.path.join(sub_path, file)
                events_file = file.split(".")[0].split("_")[0] + "_" + file.split(".")[0].split("_")[1] + "_events.tsv"
                events_file_path = os.path.join(sub_path, events_file)
                
                print(vhdr_path)
                raw = mne.io.read_raw_brainvision(vhdr_path, preload=True)                
                raw.pick_types(eeg=True)   # only keep EEG channels
                raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=50, l_freq=0.5, h_freq=40, verbose=True)
                print()
                
                # segment and make X, y
                
                print(f"Start epoching file {file} making X, y...")
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=tmin, tmax=tmax, baseline=baseline,  #  -200ms - 800ms
                    task_id=task_id, subject_id=subject_id, 
                )
                print(f"X shape: {X.shape}, y shape: {y.shape}")  
                # X: (N, SAMPLE_RATE * (tmax - tmin), C)
                # y: (N, 4) = [disease_id, stimulus_type, subject_id, task_id]
        
                # Save to memmap
                if task_id == TASK_ID_1:
                    X_mm_msit[cur_msit_idx:cur_msit_idx+X.shape[0]] = X
                    y_mm_msit[cur_msit_idx:cur_msit_idx+y.shape[0]] = y
                    cur_msit_idx += X.shape[0]
                elif task_id == TASK_ID_2:
                    X_mm_odd[cur_odd_idx:cur_odd_idx+X.shape[0]] = X
                    y_mm_odd[cur_odd_idx:cur_odd_idx+y.shape[0]] = y
                    cur_odd_idx += X.shape[0]
                elif task_id == TASK_ID_3:
                    X_mm_srt[cur_srt_idx:cur_srt_idx+X.shape[0]] = X
                    y_mm_srt[cur_srt_idx:cur_srt_idx+y.shape[0]] = y
                    cur_srt_idx += X.shape[0]
                

                print("-----------------------\n")

            print("------------------------------------------------\n")

print("DONE: cur msit =", cur_msit_idx, " expected N_total_msit =", N_total_msit)
print("DONE: cur oddball =", cur_odd_idx, " expected N_total_oddball =", N_total_oddball)
print("DONE: cur srt =", cur_srt_idx, " expected N_total_srt =", N_total_srt)

del X_mm_msit, y_mm_msit, X_mm_odd, y_mm_odd, X_mm_srt, y_mm_srt  # flush memmap to disk

------------------------------------------------

------------------------------------------------

NSERP/sub-01\eeg/
NSERP/sub-01\eeg/sub-01_task-msit_eeg.vhdr
Extracting parameters from NSERP/sub-01\eeg/sub-01_task-msit_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 1437859  =      0.000 ...  1437.859 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
✔ Cleaned: Removed 'boundary' or 'segment' annotations
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✔ Step 1: Picked common channels (123): ['FC6', 'PO10', 'F2', 'AFF2h', 'F6', 'FCC5h', 'FC2', 'TP8', 'FCC2h', 'POO2', 'F9', 'CP4', 'P6', 'CP5', 'C6', 'F10', 'POO10h', 'Fz', 'POO1', 'POz', 'AF7', 'OI1h', 'AFp1', 'FFC5h', 'AFz', 'CP2', 'AF3', 'CCP2h', 'PPO9h', 'O1', 'POO9h', 'PO8', 'Fp1', 'P2', 'FTT10h', 'Oz', 'P8', 'CP3', 'FCC1h', 'TTP8h', 'Fp2', 'Cz', 'CP6', 'FT7', 'AFF5h', 'FTT8h', 'CP1', 'PO3', 'CPP6h', 'FTT7h', 'PPO1h', 'P5', 'FC4', 'PO4', 'FC3', 'CPP4h', 


KeyboardInterrupt



## Create JSON file

In [35]:
meta_msit = {
    "N": int(N_total_msit),
    "T": SEG_LEN_MSIT,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN_1,
    "TMAX": TMAX_1,
    "BASELINE": BASELINE_1,
    "MONTAGE": "standard_1005",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "unknown"},
        "stimulus_type": {0: "F0", 1: "FS", 2: "00", 3: "S0"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {4: "MSIT"}
    }
}
with open(meta_path_msit, "w") as f:
    json.dump(meta_msit, f, indent=2)

print("Saved meta:", meta_path_msit)

Saved meta: Processed\200Hz\NSERP-MSIT\meta.json


In [36]:
meta_odd = {
    "N": int(N_total_oddball),
    "T": SEG_LEN_ODD,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN_2,
    "TMAX": TMAX_2,
    "BASELINE": BASELINE_2,
    "MONTAGE": "standard_1005",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "unknown"},
        "stimulus_type": {0: "standard", 1: "target", 2: "novel"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {1: "oddball"}
    }
}
with open(meta_path_odd, "w") as f:
    json.dump(meta_odd, f, indent=2)

print("Saved meta:", meta_path_odd)

Saved meta: Processed\200Hz\NSERP-ODD\meta.json


In [37]:
meta_srt = {
    "N": int(N_total_srt),
    "T": SEG_LEN_SRT,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN_3,
    "TMAX": TMAX_3,
    "BASELINE": BASELINE_3,
    "MONTAGE": "standard_1005",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "unknown"},
        "stimulus_type": {0: "only one type in SRT"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {9: "SRT"}
    }
}
with open(meta_path_srt, "w") as f:
    json.dump(meta_srt, f, indent=2)
    
print("Saved meta:", meta_path_srt)

Saved meta: Processed\200Hz\NSERP-SRT\meta.json


## Load and check the processed data

In [38]:
# load meta information
meta_path = meta_path_msit  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path_msit, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path_msit, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 16729
T: 300
C: 123
SAMPLE_RATE: 200
TMIN: -0.5
TMAX: 1.0
BASELINE: [-0.5, 0]
MONTAGE: standard_1005
CHANNELS: ['FC6', 'PO10', 'F2', 'AFF2h', 'F6', 'FCC5h', 'FC2', 'TP8', 'FCC2h', 'POO2', 'F9', 'CP4', 'P6', 'CP5', 'C6', 'F10', 'POO10h', 'Fz', 'POO1', 'POz', 'AF7', 'OI1h', 'AFp1', 'FFC5h', 'AFz', 'CP2', 'AF3', 'CCP2h', 'PPO9h', 'O1', 'POO9h', 'PO8', 'Fp1', 'P2', 'FTT10h', 'Oz', 'P8', 'CP3', 'FCC1h', 'TTP8h', 'Fp2', 'Cz', 'CP6', 'FT7', 'AFF5h', 'FTT8h', 'CP1', 'PO3', 'CPP6h', 'FTT7h', 'PPO1h', 'P5', 'FC4', 'PO4', 'FC3', 'CPP4h', 'CPP5h', 'PPO10h', 'F1', 'CCP5h', 'T7', 'C1', 'AF4', 'FFT7h', 'Pz', 'TPP7h', 'FFT9h', 'CPP1h', 'AF8', 'FCC6h', 'T8', 'AFp2', 'PPO5h', 'PO9', 'FFC6h', 'F3', 'C4', 'P1', 'F7', 'TP7', 'FFC4h', 'C5', 'FFC2h', 'P9', 'FT10', 'FCC4h', 'PPO2h', 'FC5', 'TPP8h', 'P7', 'FT9', 'TPP10h', 'CCP4h', 'FFT10h', 'CPz', 'F5', 'FFC1h', 'FC1', 'CPP2h', 'P10', 'CPP3h', 'FFC3h', 'PO7', 'AFF1h', 'P4', 'C2', 'OI2h', 'CCP3h', 'PPO6h', 'P3', 'TTP7h', 'CCP1h', 'FFT8h', 'CCP6h', 'FT8

In [39]:
# load meta information
meta_path = meta_path_odd  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path_odd, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path_odd, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 27865
T: 300
C: 123
SAMPLE_RATE: 200
TMIN: -0.5
TMAX: 1.0
BASELINE: [-0.5, 0]
MONTAGE: standard_1005
CHANNELS: ['FC6', 'PO10', 'F2', 'AFF2h', 'F6', 'FCC5h', 'FC2', 'TP8', 'FCC2h', 'POO2', 'F9', 'CP4', 'P6', 'CP5', 'C6', 'F10', 'POO10h', 'Fz', 'POO1', 'POz', 'AF7', 'OI1h', 'AFp1', 'FFC5h', 'AFz', 'CP2', 'AF3', 'CCP2h', 'PPO9h', 'O1', 'POO9h', 'PO8', 'Fp1', 'P2', 'FTT10h', 'Oz', 'P8', 'CP3', 'FCC1h', 'TTP8h', 'Fp2', 'Cz', 'CP6', 'FT7', 'AFF5h', 'FTT8h', 'CP1', 'PO3', 'CPP6h', 'FTT7h', 'PPO1h', 'P5', 'FC4', 'PO4', 'FC3', 'CPP4h', 'CPP5h', 'PPO10h', 'F1', 'CCP5h', 'T7', 'C1', 'AF4', 'FFT7h', 'Pz', 'TPP7h', 'FFT9h', 'CPP1h', 'AF8', 'FCC6h', 'T8', 'AFp2', 'PPO5h', 'PO9', 'FFC6h', 'F3', 'C4', 'P1', 'F7', 'TP7', 'FFC4h', 'C5', 'FFC2h', 'P9', 'FT10', 'FCC4h', 'PPO2h', 'FC5', 'TPP8h', 'P7', 'FT9', 'TPP10h', 'CCP4h', 'FFT10h', 'CPz', 'F5', 'FFC1h', 'FC1', 'CPP2h', 'P10', 'CPP3h', 'FFC3h', 'PO7', 'AFF1h', 'P4', 'C2', 'OI2h', 'CCP3h', 'PPO6h', 'P3', 'TTP7h', 'CCP1h', 'FFT8h', 'CCP6h', 'FT8

In [40]:
# load meta information
meta_path = meta_path_srt  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path_srt, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path_srt, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 5288
T: 200
C: 123
SAMPLE_RATE: 200
TMIN: -0.2
TMAX: 0.8
BASELINE: [-0.2, 0]
MONTAGE: standard_1005
CHANNELS: ['FC6', 'PO10', 'F2', 'AFF2h', 'F6', 'FCC5h', 'FC2', 'TP8', 'FCC2h', 'POO2', 'F9', 'CP4', 'P6', 'CP5', 'C6', 'F10', 'POO10h', 'Fz', 'POO1', 'POz', 'AF7', 'OI1h', 'AFp1', 'FFC5h', 'AFz', 'CP2', 'AF3', 'CCP2h', 'PPO9h', 'O1', 'POO9h', 'PO8', 'Fp1', 'P2', 'FTT10h', 'Oz', 'P8', 'CP3', 'FCC1h', 'TTP8h', 'Fp2', 'Cz', 'CP6', 'FT7', 'AFF5h', 'FTT8h', 'CP1', 'PO3', 'CPP6h', 'FTT7h', 'PPO1h', 'P5', 'FC4', 'PO4', 'FC3', 'CPP4h', 'CPP5h', 'PPO10h', 'F1', 'CCP5h', 'T7', 'C1', 'AF4', 'FFT7h', 'Pz', 'TPP7h', 'FFT9h', 'CPP1h', 'AF8', 'FCC6h', 'T8', 'AFp2', 'PPO5h', 'PO9', 'FFC6h', 'F3', 'C4', 'P1', 'F7', 'TP7', 'FFC4h', 'C5', 'FFC2h', 'P9', 'FT10', 'FCC4h', 'PPO2h', 'FC5', 'TPP8h', 'P7', 'FT9', 'TPP10h', 'CCP4h', 'FFT10h', 'CPz', 'F5', 'FFC1h', 'FC1', 'CPP2h', 'P10', 'CPP3h', 'FFC3h', 'PO7', 'AFF1h', 'P4', 'C2', 'OI2h', 'CCP3h', 'PPO6h', 'P3', 'TTP7h', 'CCP1h', 'FFT8h', 'CCP6h', 'FT8'